# 第10回 主成分の地図を読む


---


## 前回からの接続


前回は、顔の画像を主成分分析（principal component analysis）で縮め、主成分（principal component）の向きを顔として描いて読みました。第1主成分は顔全体の明るさ、第2主成分は光の当たり方でした。画像なら、向きを絵にして読めます。

けれども、燃費や重さのように、単位も意味も違う列が並んだ表では、向きを絵にしても何も見えません。今日は、そのような表の主成分の向きと得点を、どう読めばよいかを考えます。


---


## 今日の分析目標


あなたは自動車雑誌の編集者です。単位も意味も違う6つの性能の列を、2本の軸の地図にしたとき、軸と地図をどう読めばよいかに答える。

1970〜1982年に売られた392台の車の性能表を使います。単位で主成分分析の答えが変わること、軸を負荷量で読んで名前を付けること、地図に何本の軸を使うか、地図の上で1台を説明すること、を順に確かめます。

もう1つのねらいは、関数に分かれたスクリプトを、役割・入力・出力で読み、引数を変えて使えるようになることです。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、自動車雑誌の編集者です。次の号で、昔の車の特集を組みます。燃費、排気量、馬力、重さなどが並んだ性能表は、数が多すぎて、読者には読みにくい。そこで、性能表を1枚の地図にして、「この車はこういう車」と説明したいと考えています。

主成分分析を担当した同僚が、地図を作るスクリプトを残してくれました。そのスクリプトを読んで使い、地図をどう読めばよいかを考えます。


### 手元のデータ

米国のカーネギーメロン大学に集められた、1970〜1982年に売られた398台の車の性能表です（UCIのAuto MPG。Quinlan, 1993。seabornが配布しています）。1行が1台です。

列は、燃費・気筒数・排気量・馬力・重さ・加速（0から時速60マイルまでの秒数。大きいほど加速がゆるやか）・年式・地域（アメリカ・日本・ヨーロッパ）・車名です。単位は米国の単位（燃費はマイル/ガロン、排気量は立方インチ、重さはポンド）なので、スクリプトの中で日本の単位に直します。


### 最後に出す答え

地図の2本の軸の名前と、その地図が元の性能表をどれだけ残しているか、そして地図の上の1台を、軸の名前で説明する文です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 関数に分かれたスクリプトを、役割・入力・出力で読み、引数を変えて使える
- 単位がまちまちな列で、標準化（standardization）しないと主成分分析の答えが単位で変わることを説明できる
- 負荷量（得点を作るときの各列の重み）から、軸に名前を付けられる。負荷量と相関の違いを説明できる
- バイプロットで、点と矢印を合わせて読み、矢印の角度と長さの意味を説明できる
- 寄与率（proportion of variance explained）・累積寄与率（cumulative proportion of variance explained）・固有値（eigenvalue）から、地図に使う軸の本数を判断できる
- 地図の上の1台の位置を、標準化した値と負荷量の積で計算し、軸の名前で説明できる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


### データの置き場所

- 役割：データの置き場所を決める
- 入力：なし
- 出力：なし


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、③ 結果の解釈（地図をどう読むか）を中心に進みます。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

次の6つの小さな問いに答えていきます。

1. 配られたスクリプトは何をするか？（1節）
2. 単位で答えは変わるか？（2節）
3. 軸は何を表すか？（3節）
4. 車はどこにいるか？（4節）
5. 何本の軸を残すか？（5節）
6. 地図の上で1台をどう説明するか？（6節）


---


## 1. 配られたスクリプトは何をするか？


準備ができました。まず、同僚が残したスクリプトを実行し、何をする関数が入っているかを読みます。


### 配られたスクリプト：3つの関数

- 役割：性能表を読む関数`load_cars`、主成分分析をする関数`fit_pca`、地図を描く関数`biplot`を用意する
- 入力：なし
- 出力：なし（関数が使えるようになるだけ）

長いセルですが、いまは実行するだけでかまいません。1つずつ、役割・入力・出力を読んでいきます。`"""`で囲んだ1行目は、関数の説明（docstring）です。関数を読むときは、まずこの説明を読みます。


In [ ]:
def load_cars(path):
    """車の性能表を読み、日本の単位に直す。馬力が欠測の車は除く"""
    raw = pd.read_csv(path)
    raw = raw.dropna(subset=['horsepower']).reset_index(drop=True)
    region = {'usa': 'アメリカ', 'japan': '日本',
              'europe': 'ヨーロッパ'}
    cars = pd.DataFrame({
        '燃費': raw['mpg'] * 0.4251,           # マイル/ガロン → km/L
        '気筒数': raw['cylinders'],
        '排気量': raw['displacement'] * 16.39,  # 立方インチ → cc
        '馬力': raw['horsepower'],
        '重さ': raw['weight'] * 0.4536,         # ポンド → kg
        '加速': raw['acceleration'],    # 0〜60マイル毎時の秒数
        '年式': raw['model_year'] + 1900,
        '地域': raw['origin'].map(region),
        '車名': raw['name']})
    return cars


def fit_pca(cars, cols, scale=True):
    """選んだ列で主成分分析をする。scale=True なら標準化してから"""
    X = cars[cols].values
    if scale:
        X = StandardScaler().fit_transform(X)
    pca = PCA().fit(X)
    return pca, pca.transform(X)


def biplot(scores, pca, cols, groups):
    """第1・第2主成分の得点を点で、各列を矢印で、1枚に重ねて描く"""
    sd = scores.std(axis=0)                     # 得点の標準偏差
    arrows = pca.components_[:2].T * sd[:2]     # 列と得点の相関
    for g in ['アメリカ', '日本', 'ヨーロッパ']:
        m = groups == g
        plt.scatter(scores[m, 0], scores[m, 1], s=12, alpha=0.5,
                    label=g)
    for (x, y), name in zip(arrows * 3, cols):
        plt.arrow(0, 0, x, y, color='black', head_width=0.08)
        plt.text(x * 1.2, y * 1.2, name, ha='center', fontsize=12)
    plt.axhline(0, color='gray', lw=0.5)
    plt.axvline(0, color='gray', lw=0.5)
    plt.xlabel('第1主成分')
    plt.ylabel('第2主成分')
    plt.legend()
    plt.show()


### 1つ目：load_cars

- 役割：車の性能表を読み、日本の単位に直す。馬力が欠測（missing data）の車は除く
- 入力：ファイルの場所`path`
- 出力：日本語の列名の表`cars`

中身は、`read_csv`で読み、`dropna`で馬力の欠測の行を除き（第3回）、列ごとに係数を掛けて単位を直しています。たとえば、1立方インチは16.39 ccなので、排気量に16.39を掛けます。地域は`map`で日本語に直しています。


### お手本：load_carsを呼ぶ

- 役割：性能表を読み、大きさと先頭の3台を見る
- 入力：データの場所
- 出力：表の大きさと、先頭の3台


In [ ]:
cars = load_cars(f'{DATA_DIR}/auto_mpg.csv')
print(cars.shape)
cars.head(3).round(1)


392台 × 9列です。元の表は398台で、馬力が欠測の6台を除きました。欠測は398台のうち1.5%なので、この回では除いて進めます。


In [ ]:
print(round(6 / 398 * 100, 1), '%')


### 変えてみる：列の平均と標準偏差

- 役割：性能の6列の平均と標準偏差（standard deviation）を見る
- 入力：表`cars`
- 出力：平均と標準偏差の表

性能の6列の名前を`FEAT`に入れておきます。


In [ ]:
FEAT = ['燃費', '気筒数', '排気量', '馬力', '重さ', '加速']
cars[FEAT].describe().loc[['mean', 'std']].round(1)


平均は、燃費が10 km/L、排気量が3186.4 cc、重さが1350.6 kgです。標準偏差は、排気量が1715.1、重さが385.3と大きく、燃費は3.3、気筒数は1.7と小さい。列によって、数の大きさがまったく違います。


### 2つ目：fit_pca

- 役割：選んだ列で主成分分析をする。`scale=True`なら標準化してから
- 入力：表`cars`、列の名前のリスト`cols`、標準化するかどうか`scale`
- 出力：主成分分析の結果`pca`と、得点の表

`scale=True`は、引数の既定値です。呼ぶときに`scale`を書かなければ`True`になり、`scale=False`と書けば標準化しません。関数の中の`if scale:`が、その切り替えです。

最後の`return pca, pca.transform(X)`は、2つの値を返します。受け取る側は`pca, S = fit_pca(...)`のように、2つの名前を並べて受け取ります。


### お手本：fit_pcaを呼ぶ

- 役割：6列を標準化して主成分分析をし、得点の大きさと、最初の2本の寄与率を見る
- 入力：表`cars`と列`FEAT`
- 出力：得点の大きさと寄与率


In [ ]:
pca, S = fit_pca(cars, FEAT)
print(S.shape)
print(pca.explained_variance_ratio_[:2].round(3))


得点は392台 × 6本です。第1主成分の寄与率は0.798、第2主成分は0.121です。


### 3つ目：biplot

- 役割：第1・第2主成分の得点を点で、各列を矢印で、1枚に重ねて描く
- 入力：得点`scores`、主成分分析の結果`pca`、列の名前`cols`、色分けに使う区分`groups`
- 出力：図（返す値はない）

中身は、4節で読みます。いまは呼ぶだけです。


### お手本：biplotを呼ぶ

- 役割：392台を、地域で色分けして地図に描く
- 入力：得点`S`、`pca`、列`FEAT`、地域の並び
- 出力：地図


In [ ]:
biplot(S, pca, FEAT, cars['地域'].values)


3つの関数を呼ぶだけで、地図ができました。点が1台ずつの車、矢印が性能の列です。アメリカ車は右に、日本車とヨーロッパ車は左に集まっています。

けれども、この地図をそのまま読者に見せても、横の軸と縦の軸が何を表すのかが分かりません。そもそも、標準化してよかったのでしょうか。次の節から、順に確かめます。


---


## 2. 単位で答えは変わるか？


`fit_pca`は、標準化してから主成分分析をしていました。標準化しないと、何が起きるでしょうか。引数`scale=False`で確かめます。


### お手本：標準化しないで主成分分析

- 役割：6列を標準化せずに主成分分析をし、寄与率と第1主成分の負荷量を見る
- 入力：表`cars`と列`FEAT`
- 出力：最初の2本の寄与率と、第1主成分の向き

`pca.components_[0]`は、第1主成分の向き（第9回）です。6列それぞれの数が並びます。


In [ ]:
pca_raw, S_raw = fit_pca(cars, FEAT, scale=False)
print(pca_raw.explained_variance_ratio_[:2].round(3))
print(pd.Series(pca_raw.components_[0], index=FEAT).round(3))


第1主成分の寄与率は0.994です。1本で99%以上を受け持っています。第1主成分の向きは、排気量が0.978で、ほかはほとんど0です。第1主成分は、ほぼ排気量そのものです。


### なぜ排気量だけになるのか

主成分分析は、平均からのずれの二乗和を、いちばん多く受け持つ向きを探します（第9回）。排気量はccで測るので、標準偏差が1715.1もあります。燃費（3.3）や気筒数（1.7）のずれは、排気量のずれに比べて小さすぎて、ほとんど効きません。

第1主成分が排気量になったのは、排気量が車の性能でいちばん大事だからではありません。ccで測ると、数が大きいからです。


### TODO①：排気量を立方インチに戻すと

表`cars`をコピーした`cars_ci`を作り、排気量の列を16.39で割って立方インチに戻してください。そのうえで、標準化しない主成分分析をし、最初の2本の寄与率と、第1主成分の向きを表示してください。


In [ ]:
# TODO: 排気量を立方インチに戻し、標準化しない主成分分析をする
# ヒント: cars_ci = cars.copy()
#         cars_ci['排気量'] = cars['排気量'] / 16.39
#         pca_ci, S_ci = fit_pca(cars_ci, FEAT, scale=False)
...


立方インチに戻すと、第1主成分の向きは重さが0.965になり、排気量は0.247に下がります。寄与率は0.989です。同じ392台なのに、排気量の単位を変えただけで、第1主成分は「排気量の軸」から「重さの軸」に入れ替わりました。立方インチにすると排気量の数は16.39分の1になり、kgの重さ（標準偏差385.3）のほうが、数が大きくなるからです。


### お手本：標準化すると

標準化は、各列から平均を引き、標準偏差で割る変換でした（第4回・第7回）。標準化した列は、どれも標準偏差が1になり、単位が消えます。

- 役割：ccのままの表と、立方インチに戻した表で、標準化してから主成分分析をし、寄与率を比べる
- 入力：表`cars`と`cars_ci`（`cars_ci`は、ここで作り直す）
- 出力：2つの寄与率


In [ ]:
cars_ci = cars.copy()
cars_ci['排気量'] = cars['排気量'] / 16.39
pca_cc, _ = fit_pca(cars, FEAT)
pca_ci, _ = fit_pca(cars_ci, FEAT)
print(pca_cc.explained_variance_ratio_[:2].round(3))
print(pca_ci.explained_variance_ratio_[:2].round(3))


標準化すると、ccでも立方インチでも、寄与率は0.798と0.121で、同じです。単位によらない答えになりました。`_`は、受け取るけれども使わない値の置き場です。

単位も意味も違う列が並んだ表では、標準化してから主成分分析をします。同僚の`fit_pca`の既定値が`scale=True`だったのは、このためです。逆に、前回の顔の画像のように、どの列も同じ単位（画素の明るさ）なら、標準化しなくてもかまいません。


---


## 3. 軸は何を表すか？


標準化して、単位によらない地図ができました。次は、地図の2本の軸が何を表すかを読みます。


### 得点は、標準化した列の重み付きの和

第9回で、得点は「ずれ`@`向きの転置」でした。標準化した列を$z_{i1}, \dots, z_{i6}$、第1主成分の向きの6つの成分を$a_{11}, \dots, a_{61}$と書くと、$i$台目の第1主成分の得点は

$$t_{i1} = a_{11} z_{i1} + a_{21} z_{i2} + \cdots + a_{61} z_{i6}$$

です。向きの成分$a_{j1}$は、得点を作るときの、列$j$の重み（weight）です。この重みを、主成分負荷量、または負荷量（loading）と呼びます。重みが大きい列ほど、その軸に強く効きます。


### お手本：負荷量の表

- 役割：第1・第2主成分の負荷量を、列の名前を付けた表にする
- 入力：`pca.components_`
- 出力：6行2列の表

`pca.components_[:2]`は2行6列（2本の向き）なので、`.T`で転置して6行2列にします。


In [ ]:
load = pd.DataFrame(pca.components_[:2].T, index=FEAT,
                    columns=['第1', '第2'])
load.round(3)


In [ ]:
load.plot.barh(subplots=True, layout=(1, 2), figsize=(10, 3.5),
               legend=False, sharey=True)
plt.show()


`plot.barh`は、表の列ごとに横棒を描きます。`subplots=True`で、第1と第2を別の図に分けています。


### 第1主成分：大きさ・パワーの軸

第1主成分では、気筒数・排気量・馬力・重さの負荷量が0.43〜0.44で、ほぼ同じ大きさです。燃費は−0.399、加速は−0.292で、逆の符号です。

気筒数が多く、排気量が大きく、馬力が強く、重い車ほど、第1主成分の得点が大きくなります。そのような車は、燃費が悪く、加速の秒数が短い（すばやく加速する）車です。第1主成分は「車の大きさ・パワーの軸」と名付けられます。


### 第2主成分：加速のゆるやかさの軸

第2主成分では、加速の負荷量が0.893で、飛び抜けて大きい。ほかの列は0.3より小さい。第2主成分の得点が大きいのは、時速60マイルまでに時間がかかる、加速のゆるやかな車です。第2主成分は「加速のゆるやかさの軸」と名付けられます。

軸の名前は、負荷量を見て人が付けるあだ名です。データが名前を教えてくれるわけではありません。読者に伝わる言葉を選びます。


### 符号は決まらない

第9回で見たとおり、向きは逆向きにしても同じ直線です。負荷量の符号がすべて逆になっても、地図は左右（または上下）が裏返るだけです。読むべきは、どの列が同じ符号で並び、どの列が逆の符号かです。


### 負荷量と相関

負荷量は、得点を作るときの重みでした。これとは別に、列と得点の相関係数も、軸を読む手がかりになります。標準化した列と得点の相関は、負荷量に、その軸の得点の標準偏差（$\sqrt{固有値}$）を掛けたものになります。

$$\text{列}j\text{と第}k\text{主成分の得点の相関} = a_{jk} \times (\text{第}k\text{主成分の得点の標準偏差})$$

- 役割：負荷量の表に、得点の標準偏差を掛けて、相関の表を作る
- 入力：負荷量`load`、得点`S`
- 出力：6行2列の相関の表

`S.std(axis=0)`は、得点の列ごと（軸ごと）の標準偏差です。


In [ ]:
sd = S.std(axis=0)
print(sd[:2].round(3))
(load * sd[:2]).round(3)


第1主成分の得点の標準偏差は2.188、第2主成分は0.854です。相関で見ると、第1主成分と排気量の相関は0.971、第2主成分と加速の相関は0.762です。

同じ軸の中では、負荷量と相関の大小の順は同じです（同じ数を掛けただけだから）。軸どうしを比べるときは、相関のほうが分かりやすいことがあります。本によっては、この相関のほうを負荷量と呼ぶこともあるので、読むときは定義を確かめましょう。


### TODO②：相関を直接確かめる

`np.corrcoef`で、元の排気量の列`cars['排気量']`と第1主成分の得点`S[:, 0]`の相関係数を出し、表の0.971と同じになるかを確かめてください。同じく、加速と第2主成分の得点`S[:, 1]`の相関係数も出してください。


In [ ]:
# TODO: 排気量と第1主成分、加速と第2主成分の相関係数を出す
# ヒント: np.corrcoef(cars['排気量'], S[:, 0])[0, 1]
...


排気量と第1主成分の相関は0.971、加速と第2主成分の相関は0.762で、表と同じです。標準化は、引いて割るだけの変換なので、元の列で相関を出しても、標準化した列で出しても同じになります。


---


## 4. 車はどこにいるか？


2本の軸に名前が付きました。では、392台の車は、地図のどこにいるでしょうか。1節の地図を、軸の名前を手がかりに読み直します。


### biplotの中身を読む

`biplot`の中身を、かたまりで読みます。

1. `sd = scores.std(axis=0)`と`arrows = pca.components_[:2].T * sd[:2]`：3節の相関の表を作る。矢印の先は、列と第1・第2主成分の相関
2. 1つ目の`for`：地域ごとに、得点を散布図にする
3. 2つ目の`for`：列ごとに、原点から矢印を引き、名前を書く。矢印は図に合わせて3倍に伸ばしてある

点と矢印を1枚に重ねた図を、バイプロット（biplot）と呼びます。


### お手本：バイプロットを読み直す

- 役割：1節と同じ地図をもう一度描き、地域ごとの得点の平均を出す
- 入力：得点`S`、`pca`、地域の並び
- 出力：地図と、地域ごとの平均


In [ ]:
biplot(S, pca, FEAT, cars['地域'].values)
pd.DataFrame(S[:, :2], columns=['第1', '第2']).groupby(
    cars['地域']).mean().round(2)


アメリカ車は、第1主成分の平均が1.03で、右（大きく、パワーのある側）にいます。日本車は−1.82、ヨーロッパ車は−1.58で、左（小さく、燃費のよい側）にいます。第2主成分の平均は、どの地域も0に近く、地域による違いは小さい。

矢印の向きにある点ほど、その列の値が大きい車です。右に伸びる気筒数・排気量・馬力・重さの矢印の先にアメリカ車が、左に伸びる燃費の矢印の先に日本車とヨーロッパ車がいます。


### 矢印の角度は、相関の目安

矢印どうしの角度も読めます。同じ向きの矢印の列は相関が正、逆向きの矢印の列は相関が負、直角に近ければ相関は0に近い、という目安です。燃費の矢印は、重さの矢印とほぼ逆を向いています。


### TODO③：相関の表と比べる

`cars[FEAT].corr()`で6列の相関係数の表を出し、燃費と重さ、加速と馬力の相関係数を、矢印の角度と比べてください。


In [ ]:
# TODO: 6列の相関係数の表を出し、矢印の角度と比べる
# ヒント: cars[FEAT].corr().round(2)
...


燃費と重さの相関は−0.83で、矢印はほぼ逆を向いています。加速と馬力の相関は−0.69で、矢印は鈍い角度で開いています。気筒数・排気量・重さどうしは0.9以上で、矢印はほぼ重なっています。地図の矢印の角度は、相関の表とおおよそ合っています。


### 矢印の長さは、残せている割合

矢印の先は、列と第1・第2主成分の相関でした。標準化した列の散らばりのうち、2本の軸で表せている割合は、2つの相関の二乗の和になります。矢印が長い列ほど、地図の上でよく表せています。

- 役割：6列それぞれの矢印の長さの二乗（2本の軸で残せている割合）を出す
- 入力：相関の表`load * sd[:2]`
- 出力：6列の割合

`** 2`で二乗し、`sum(axis=1)`で横の方向（2本の軸）に足します（第9回）。


In [ ]:
((load * sd[:2]) ** 2).sum(axis=1).round(3)


加速は0.989、排気量は0.951と、ほぼ全部を表せています。いちばん小さい燃費でも0.806です。どの列も、2本の地図で8割以上が残っています。逆に言えば、燃費の2割ほどは、この地図には表れていません。2本の軸の地図は、元の6列の近似です。


---


## 5. 何本の軸を残すか？


2本の地図で、どの列も8割以上が残っていました。では、軸を3本、4本と増やすべきでしょうか。地図に使う本数を、寄与率と固有値から考えます。


### お手本：6本すべての寄与率

- 役割：6本すべての寄与率と累積寄与率を出し、棒と線で描く
- 入力：`pca`
- 出力：寄与率、累積寄与率、図

寄与率を軸の順に並べた図を、スクリープロット（scree plot）と呼びます。


In [ ]:
ratio = pca.explained_variance_ratio_
print(ratio.round(3))
print(ratio.cumsum().round(3))
plt.bar(range(1, 7), ratio)
plt.plot(range(1, 7), ratio.cumsum(), 'o-', color='#e63946')
plt.xlabel('主成分')
plt.ylabel('寄与率')
plt.show()


第1主成分が0.798、第2主成分が0.121で、第3主成分からは0.043、0.021と、ほとんど残りません。累積寄与率は、2本で0.919、3本で0.963です。


### 本数の目安

本数の決め方には、いくつかの目安があります。

- 累積寄与率：たとえば9割を超えるまで残す（第9回）。ここでは2本
- スクリープロットの肘：棒が急に低くなる手前まで残す。ここでは、第1主成分のあとで急に低くなる
- 固有値1以上：標準化したとき、固有値（得点の分散）が1以上の軸を残す。元の1列ぶんの分散より多くを受け持つ軸だけを残す、という考え方


### TODO④：固有値が1以上の軸の数

固有値は`pca.explained_variance_`に入っています。固有値を表示し、1以上の軸の数を数えてください。


In [ ]:
# TODO: 固有値を表示し、1以上の軸の数を数える
# ヒント: print(pca.explained_variance_.round(3))
#         (pca.explained_variance_ >= 1).sum()
...


固有値は4.801、0.73、0.259…で、1以上は第1主成分の1本だけです。固有値1以上の目安では1本、累積寄与率9割の目安では2本と、目安が食い違いました。

目安は、目的に合わせて選ぶものです。今日は、読者に見せる地図を作りたいので、2本にします。2本なら平面の地図に描け、累積寄与率は0.919で、どの列も8割以上が残っていました。第2主成分の寄与率は0.121と小さいですが、加速という、第1主成分では表しきれない性能を受け持っています。


---


## 6. 地図の上で1台をどう説明するか？


2本の軸の地図に決めました。最後に、地図の上の1台を、軸の名前で読者に説明します。まず、1台の位置を自分で計算して、地図の点が何を表すかを確かめます。


### お手本：カローラの位置を手で計算する

- 役割：トヨタ・カローラ1200を探し、6列を標準化して、負荷量との積で得点を計算する。`fit_pca`の得点と比べる
- 入力：表`cars`、`pca`、得点`S`
- 出力：カローラの行番号、手で計算した得点、`fit_pca`の得点

`std(ddof=0)`は、`StandardScaler`と同じ標準偏差の計算です（第7回）。`z.values @ pca.components_[:2].T`は、標準化した6つの値と、2本の向きの積です（第9回の「ずれ`@`向きの転置」）。


In [ ]:
i = cars.index[cars['車名'] == 'toyota corolla 1200'][0]
z = (cars.loc[i, FEAT] - cars[FEAT].mean()) / cars[FEAT].std(ddof=0)
print(i)
print((z.values.astype(float) @ pca.components_[:2].T).round(2))
print(S[i, :2].round(2))


カローラ1200は52行目で、手で計算した得点は（−2.71, 0.39）です。`fit_pca`の得点と同じです。地図の点は、標準化した性能を、負荷量の重みで足した位置でした。

`.astype(float)`は、表から1行を取り出すと数の種類が混ざった並びになるので、計算できる小数の並びにそろえる指定です。


### お手本：2台を性能表と並べる

- 役割：カローラ1200とシボレー・インパラの、性能と得点を並べる
- 入力：表`cars`と得点`S`
- 出力：2台の表


In [ ]:
two = cars['車名'].isin(['toyota corolla 1200', 'chevrolet impala'])
show = cars.loc[two, FEAT + ['年式', '地域', '車名']].round(1)
show['第1'] = S[two, 0].round(2)
show['第2'] = S[two, 1].round(2)
show.drop_duplicates('車名')


カローラ1200（1971年、日本）は、燃費13.2 km/L、排気量1163.7 cc、重さ804.2 kgで、第1主成分は−2.71です。インパラ（1970年、アメリカ）は、燃費6 km/L、排気量7441.1 cc、重さ1975 kgで、第1主成分は4.92です。インパラは、年式の違う同じ車名の車が表に何台かあるので、`drop_duplicates`で最初の1台だけを残しました。

読者には、こう説明できます。「カローラ1200は、地図の左端近く、小さくて燃費のよい車の代表です。インパラは右端近く、大きくてパワーのある車の代表です。第2主成分で見ると、インパラは−1.37で、すばやく加速する側にいます。」


### TODO⑤：地図のいちばん上の車

第2主成分の得点がいちばん大きい車（加速がいちばんゆるやかな側の車）を探し、車名・年式・地域・加速と、2つの得点を表示してください。


In [ ]:
# TODO: 第2主成分の得点がいちばん大きい車を探す
# ヒント: j = np.argmax(S[:, 1])
#         print(cars.loc[j, ['車名', '年式', '地域', '加速']])
#         print(S[j, :2].round(2))
...


地図のいちばん上は、プジョー504（1979年、ヨーロッパ）です。加速は24.8秒で、時速60マイルまでに25秒近くかかります。第2主成分は2.91、第1主成分は−2.04です。小さめで、加速のとてもゆるやかな車として説明できます。


### 答え

標準化した6つの性能の列を主成分分析すると、2本の軸で、元の散らばりの91.9%を表せます。

- 横の軸（第1主成分、寄与率0.798）は、気筒数・排気量・馬力・重さが同じ符号で並び、燃費が逆の符号の「車の大きさ・パワーの軸」。右にアメリカ車、左に日本車とヨーロッパ車
- 縦の軸（第2主成分、寄与率0.121）は、加速の負荷量が0.893の「加速のゆるやかさの軸」

地図の上の1台は、2つの軸の名前で説明できます。カローラ1200は左端近く（−2.71, 0.39）の小さく燃費のよい車、インパラは右端近く（4.92, −1.37）の大きくパワーのある車です。ただし、単位がまちまちな列は、標準化してから主成分分析をする必要があります。標準化しないと、単位の選び方で、第1主成分が排気量にも重さにもなりました。


まだ言えないこと

- 地図に表れていない違い：燃費の2割ほどなど、3本目より先の軸の違いは、この地図には表れない
- 軸の名前の確かさ：名前は負荷量を見て付けたあだ名で、車を何台か入れ替えると、第2主成分の向きが変わることもありうる
- 年式の違い：今日の6列には年式を入れていない。年代による変化は、この地図では読めない


---


## 目標に答えられたか


- 今日の目標は「単位も意味も違う6つの性能の列を、2本の軸の地図にしたとき、軸と地図をどう読めばよいか」でした。2本の軸の名前と、地図が残している割合を答えられますか？
- 1節で、配られた3つの関数を、どの順に何を読んで理解しましたか？
- 2節で、標準化しないと第1主成分が排気量になったのはなぜですか？ 単位を変えると何が起きましたか？
- 3節の負荷量と、列と得点の相関は、どう違いますか？
- 4節のバイプロットで、矢印の角度と長さは、それぞれ何を表しますか？
- 5節で、本数の目安が食い違ったとき、どう決めましたか？
- 6節で、地図の上の1台の位置を、どう計算しましたか？


---


## 今日の要点


- 今日、持ち帰る1文：単位がまちまちな列は、標準化してから主成分分析をする。軸は負荷量（得点を作るときの各列の重み）で読み、名前を付ける。2本の地図は近似なので、どれだけ残せているかを寄与率と矢印の長さで確かめてから説明に使う
- 標準化しないと、数の大きい列が第1主成分を占める。単位を変えると、答えが変わる
- 負荷量は得点を作るときの重み。列と得点の相関は、負荷量×得点の標準偏差。符号そのものに意味はなく、どの列が同じ符号で並ぶかを読む
- バイプロット：点は車、矢印は列。矢印の向きの車ほどその列の値が大きい。矢印の角度は相関の目安、長さは2本で残せている割合
- 本数の目安（累積寄与率・肘・固有値1以上）は食い違うことがある。目的に合わせて選ぶ
- Python：関数は、まずdocstringと引数・返す値（役割・入力・出力）を読む。既定値のある引数（`scale=True`）は、書き換えて使える


---


## 次回へ


今日は、392台の車を、2本の軸の地図に置きました。地図の上では、アメリカ車と、日本・ヨーロッパ車が、左右に分かれて見えました。けれども、地域の列を使わなかったら、車はいくつのまとまりに分かれるでしょうか。

次回の問いは、答えの列を使わずに、データをいくつかのグループに分けるには、どうすればよいか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

今日の6列に年式を足した7列`FEAT7`で、`fit_pca`を呼び直してください。最初の3本の寄与率と、第1・第2主成分の負荷量の表を表示しましょう。関数の引数`cols`を変えるだけです。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`FEAT7 = FEAT + ['年式']`として、`pca7, S7 = fit_pca(cars, FEAT7)`とします。
負荷量の表は、3節と同じく`pd.DataFrame(pca7.components_[:2].T, index=FEAT7, columns=['第1', '第2'])`です。

</details>


### 応用②（判断）

応用①の第2主成分で、負荷量の絶対値がいちばん大きい列はどれですか。列の名前を答えてください（例: 馬力）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

負荷量の表の第2の列を`abs()`で絶対値にし、`idxmax()`で、いちばん大きい行の名前を取り出せます。

</details>


### 応用③（解釈）

雑誌の読者に向けて、今日の地図の読み方を3行程度で説明してください。横の軸と縦の軸が何を表すか、地図が元の性能表をどれだけ表しているか、地図に表れないことは何か、を入れましょう。


（ここに3行程度で書く）


---


## 発展（任意）


### t-SNEで地図を描く

主成分分析の地図は、元の6列の散らばりをできるだけ多く残す向きで描いた地図でした。これとは別に、「近くにいる車どうしを、地図の上でも近くに置く」ことだけを目指して描く方法があります。その代表がt-SNE（ティー・スニー）です。

- 役割：標準化した6列を、t-SNEで2次元の地図にし、主成分分析の地図と並べる
- 入力：表`cars`
- 出力：2枚の地図

`perplexity`は、何台くらいを「近所」とみなすかの目安です（既定値は30）。計算に数秒かかります。


In [ ]:
from sklearn.manifold import TSNE
Z = StandardScaler().fit_transform(cars[FEAT])
T = TSNE(perplexity=30, random_state=0).fit_transform(Z)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for g in ['アメリカ', '日本', 'ヨーロッパ']:
    m = (cars['地域'] == g).values
    axes[0].scatter(S[m, 0], S[m, 1], s=10, alpha=0.5, label=g)
    axes[1].scatter(T[m, 0], T[m, 1], s=10, alpha=0.5, label=g)
axes[0].set_title('主成分分析')
axes[1].set_title('t-SNE')
axes[0].legend()
plt.show()


t-SNEの地図では、車がいくつかの島に分かれて見えます。けれども、t-SNEの横と縦の軸には、主成分分析のような意味（負荷量）がありません。島どうしの離れ具合も、元の性能の違いの大きさを表すとは限りません。


### 距離が保たれているかを測る

- 役割：392台のすべての2台の組について、標準化した6列での距離と、地図の上での距離を出し、その相関係数を比べる
- 入力：標準化した6列`Z`、主成分分析の得点`S`、t-SNEの地図`T`
- 出力：2つの相関係数

`pairwise_distances`は、すべての2台の組の距離の表を返します。`np.triu_indices`で、表の右上半分（同じ組を2回数えない部分）だけを取り出します。


In [ ]:
from sklearn.metrics import pairwise_distances
D = pairwise_distances(Z)
up = np.triu_indices(len(Z), 1)
D_pca = pairwise_distances(S[:, :2])
D_tsne = pairwise_distances(T)
print(np.corrcoef(D[up], D_pca[up])[0, 1].round(3))
print(np.corrcoef(D[up], D_tsne[up])[0, 1].round(3))


主成分分析の地図は、元の距離との相関が0.993で、遠い車は地図でも遠く置かれています。t-SNEは0.913で、近所の並びを優先する代わりに、遠い車どうしの距離は保ちません。

試すなら、`perplexity=5`にして描き直し、地図の形と距離の相関がどう変わるかを見てみましょう。近所を狭くとると、島が細かく分かれ、距離の相関は下がります。さらに試すなら、`!pip install -q umap-learn`で入れたUMAP（`umap.UMAP(random_state=0).fit_transform(Z)`）でも、同じ比べ方ができます。
